**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/yang-2026-electrophysiological"], check=True)
    os.chdir("/content/encore/entries/yang-2026-electrophysiological")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Yang & Xiao (2026) on DANDI:000020

Paper. Zihan Yang and Yuchen Xiao, "Electrophysiological response geometries reveal shared and
protocol-specific cell-type organization", bioRxiv doi:10.64898/2026.01.08.698522, version 2
(2026-08-20). The claims file (`claims/yang-2026-electrophysiological.yaml`) was written from that
text, its Methods and figure legends, and Supplementary Table 1; the paper links no code.

Dataset. DANDI:000020, "Patch-seq recordings from mouse visual cortex" (Allen Institute), pinned
at version 0.210913.1639 (doi:10.48324/dandi.000020/0.210913.1639). This is the version the paper
cites (Data and Code Availability) and the only published version; it holds 4435 NWB files, one
per cell. Every file is streamed with remfile and h5py and reduced as it loads to one record per
sweep of the three stimulus families; nothing is downloaded to disk.

What the paper analysed. Current-clamp sweeps of genetically targeted cells, grouped by Cre
line: long-pulse (sustained current steps), ramp, and brief-pulse sweeps (X5SP_Search_DA_0),
quality-controlled, reduced to one feature vector per cell and protocol (Supplementary Table 1),
standardized and projected by PCA; driver-line centroids, their distances and their alignment
across protocols, with balanced resampling and label-shuffle nulls.

Method details the paper leaves open (quality-control thresholds, spike detection, the
long-pulse and ramp stimulus families, response windows, feature-set membership, the number of PCs,
the line inclusion rule and the resampling settings) are run under every reading the claims name;
the readings and the main reading adopted for each are stated in `geometry.py` and in the cell that
builds the geometries.

| Claim | Tested |
|---|---|
| fig1c-cells-per-protocol | tested |
| fig1d-driver-line-coverage | tested |
| fig2a-longpulse-low-dimensional | tested |
| fig2b-longpulse-continuous-biased | tested |
| fig2d-longpulse-loadings | tested |
| fig2c-longpulse-centroid-distances | tested |
| fig2e-longpulse-pc2-pvalb-high-vip-low | tested |
| fig3a-ramp-variance | tested |
| fig3b-ramp-continuous-biased | tested |
| fig3d-ramp-loadings | tested |
| fig3e-ramp-recruitment-current-order | tested |
| fig4a-brief-response-variance | tested |
| fig4b-brief-continuous-biased | tested |
| fig4c-brief-response-pc1-loadings | tested |
| fig4e-brief-response-base-alignment | tested |
| fig4f-brief-gain-vip-cck-high | tested |
| disc-brief-gain-sncg-cck-non-pvalb | tested |
| fig5a-longpc2-ramppc1-centroids | tested |
| fig5b-long-ramp-distance-alignment | tested |
| fig5c-balanced-correlations-positive | tested |
| fig5d-f-true-above-shuffle-null | tested |

The run takes about 20 minutes: about 10 to stream the 4435 files in 40 processes, and about 10 for
the analyses, most of it the balanced resampling under every combination of readings.

In [ ]:
import time, zlib, resource, os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
import geometry as G
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
KEY = "yang-2026-electrophysiological"
def seed(claim): return zlib.crc32(f"{KEY}/{claim}".encode())
t0 = time.time()
assets = G.list_assets()
print("dandiset", G.DANDISET, "version", G.VERSION, "assets", len(assets), "bytes", sum(a["size"] for a in assets))

In [ ]:
# Stream every NWB file and reduce it to per-sweep records (about 10 to 15 minutes on 4 CPUs).
cells, sw = G.read_all(assets, workers=40)
print(f"read {len(cells)} files in {time.time() - t0:.0f} s; files with errors: {cells.err.notna().sum()}")
print("peak memory of this process (MB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3)

## Data check

What was loaded, checked against the dandiset's description and the paper's.

In [ ]:
print("files", len(cells), "sessions", cells.session.nunique(), "subjects", cells.subject.nunique())
print("sessions per subject:", cells.groupby("subject").size().value_counts().sort_index().to_dict())
print("\nsweep records by family and stimulus description:")
print(sw.groupby(["fam", "desc"]).size().to_string())
print("\nsampling rate (Hz) by family:", sw.groupby(["fam", "rate"]).size().to_dict())
print("data unit attribute:", sw.unit.value_counts().to_dict(), "(values converted with the conversion attribute to mV)")
for fam in ["long", "ramp", "brief"]:
    s = sw[(sw.fam == fam) & sw.qc_len.fillna(False).astype(bool)]
    print(f"{fam}: baseline potential (mV) range {s.base.min():.1f} to {s.base.max():.1f}, median {s.base.median():.1f}; "
          f"baseline SD (mV) median {s.noise.median():.3f}")
print("\nlong-pulse step durations (s):", sw[sw.fam == "long"].dur.round(3).value_counts().to_dict())
print("long-pulse command amplitudes (pA): range", sw[sw.fam == "long"].amp.min(), "to", sw[sw.fam == "long"].amp.max())
r = sw[(sw.fam == "ramp") & sw.qc_len.fillna(False).astype(bool)]
print("ramp command slope (pA/s): median", round(r.slope.median(), 2), "range", round(r.slope.min(), 2), "to", round(r.slope.max(), 2),
      "; linearity R^2 min", round(r.lin_r2.min(), 5))
b = sw[sw.fam == "brief"]
print("brief pulse amplitudes (pA): range", b.amp.min(), "to", b.amp.max(), "; sweeps with amplitude <= 0:", int((b.amp <= 0).sum()))

In [ ]:
# Quality control (Methods 2; thresholds are this attempt's standard choices, see geometry.py)
for fam in ["long", "ramp", "brief"]:
    s = sw[sw.fam == fam]
    print(f"{fam}: sweeps {len(s)}, detectable stimulus {s.qc_stim.fillna(False).astype(bool).sum()}, "
          f"complete trace {(s.qc_stim.fillna(False).astype(bool) & s.qc_len.fillna(False).astype(bool)).sum()}, "
          f"pass all incl. baseline {G._qc(s).sum()}")

In [ ]:
# Recordings deposited more than once: compare per-sweep summaries (stimulus, amplitude, peak response)
def sig(g):
    g = g.sort_values("sweep")
    return tuple(zip(g.desc, g.amp.round(2).fillna(-1), g.peak.round(4).fillna(-1)))
sigs = sw.groupby("session").apply(sig)
dup = sigs[sigs.duplicated(keep=False)]
groups = dup.groupby(dup.map(hash)).groups
print("sessions whose sweep summaries are identical to another session's:", len(dup))
for k, idx in groups.items():
    print(cells.set_index("session").loc[list(idx), ["subject", "genotype"]].to_string())
print("first-sweep fingerprint duplicates:", int(cells.fingerprint.astype(str).duplicated(keep=False).sum()))
DUP_SESSIONS = set(dup.index)

In [ ]:
# Clock check: the command and the voltage are in one series per sweep. Contrast the voltage change in the
# 20 ms before the command onset with the change 5 to 25 ms after it, for negative and positive commands.
q = sw[G._qc(sw, baseline=False)]
L = q[q.fam == "long"]
for sign, m in (("negative", L.amp < 0), ("positive", L.amp > 0)):
    print(f"long pulse, {sign} commands (n={m.sum()}): mean dV before onset {L.dv_pre[m].mean():+.3f} mV, "
          f"after onset {L.dv_post[m].mean():+.2f} mV; sweeps with dV after onset of the command's sign "
          f"{np.mean(np.sign(L.dv_post[m]) == (1 if sign == 'positive' else -1)):.4f} ({int((np.sign(L.dv_post[m]) != (1 if sign == 'positive' else -1)).sum())} sweeps otherwise)")
B = q[q.fam == "brief"]
print(f"brief pulse (n={len(B)}): mean dV 5 ms before onset {B.dv_pre.mean():+.3f} mV, in 5 ms after pulse end {B.dv_post.mean():+.2f} mV")
R = q[q.fam == "ramp"]
print(f"ramp: first spike (0 mV) at command {R.rec_zero.median():.1f} pA median, latency {R.lat_zero.median():.2f} s, "
      f"command/latency ratio {np.nanmedian(R.rec_zero / R.lat_zero):.2f} pA/s (ramp slope {R.slope.median():.2f} pA/s)")

In [ ]:
# Driver lines from the genotype (Methods 1); the readings are in geometry.driver_line
cells["line"] = cells.genotype.map(G.driver_line)
cells["line_cre"] = cells.genotype.map(lambda g: G.driver_line(g, "cre"))
cells["line_flp"] = cells.genotype.map(lambda g: G.driver_line(g, "flp"))
LINE = cells.set_index("session").line.copy()
LINE[list(DUP_SESSIONS)] = None   # a recording filed under two genotypes: label unknown, excluded
print("cells per driver line (first driver allele):")
print(LINE.value_counts().head(25).to_string())
print("genotypes with no driver allele:", cells[cells.line.isna()].genotype.value_counts().to_dict())
print("cells whose driver allele is Flp only (no Cre):", cells[cells.line_cre.isna() & cells.line.notna()].line.value_counts().to_dict())
old_set = cells.descs.map(lambda d: any(k.startswith("C1") for k in d)); new_set = cells.descs.map(lambda d: any(k.startswith("X") for k in d))
print("cells with the older stimulus set (C1*):", int(old_set.sum()), "; newer (X*):", int(new_set.sum()), "; both:", int((old_set & new_set).sum()), "; older only:", int((old_set & ~new_set).sum()), "; neither:", int((~old_set & ~new_set).sum()),
      "; cells with X5SP_Search_DA_0:", int(cells.descs.map(lambda d: "X5SP_Search_DA_0" in d).sum()))
print("intersectional genotypes (Cre and Flp):", int((cells.line_cre.notna() & cells.line_flp.notna() & (cells.line_cre != cells.line_flp)).sum()))
L9 = ["Sncg", "Oxtr", "Ndnf", "Htr3a", "Gad2", "Slc32a1", "Pvalb", "Sst", "Vip"]
L11 = ["Gad2", "Htr3a", "Ndnf", "Oxtr", "Pvalb", "Sst", "Vip", "Sncg", "Cck", "Calb2", "Crh"]

## Cell-level features and geometries

Per-cell features follow Supplementary Table 1 (definitions in `geometry.py`). Each geometry is
computed under every reading the claims name. Cells enter a geometry when they have at least one
quality-controlled sweep of the family and belong to one of the driver lines the paper shows for
that protocol (9 for long pulse and ramp, Figures 2B and 3B; 11 for brief pulse, Figure 4D).

In [ ]:
QC = sw[G._qc(sw)].copy()
LONGQ = QC[(QC.fam == "long") & (QC.amp > 0)]
def lab(idx): return LINE.reindex(idx)
LONGF = {}
for fam, rx in G.LONG_FAMILIES.items():
    for spike in ("zero", "dvdt"):
        F = G.long_features(LONGQ[LONGQ.desc.str.match(rx)], spike)
        LONGF[(fam, spike)] = F[lab(F.index).isin(L9)]
RAMPF = {s: (lambda F: F[lab(F.index).isin(L9)])(G.ramp_features(QC[QC.fam == "ramp"], s)) for s in ("zero", "dvdt")}
BRIEFF = {w: (lambda F: F[lab(F.index).isin(L11)])(G.brief_features(QC[QC.fam == "brief"], w)) for w in ("w10", "w50")}
BIN = [c for c in LONGF[("all_long_square", "zero")].columns if c.startswith("bin_")]
LONG_TABLE = [c for c in LONGF[("all_long_square", "zero")].columns if not c.startswith("bin_")]
GEO = {}
for (fam, spike), F in LONGF.items():
    for fs, cols in (("a_all", list(F.columns)), ("b_nobins", LONG_TABLE)):
        for imp in ("mean", "median"):
            GEO[("long", fam, fs, imp, spike)] = G.orient(G.pca_geometry(F[cols], imp), "PC2", "first_spiking_command")
for spike, F in RAMPF.items():
    for fs, cols in (("a_fig3d", G.RAMP_RECRUIT), ("b_all", list(F.columns))):
        for imp in ("mean", "median"):
            GEO[("ramp", fs, imp, spike)] = G.orient(G.pca_geometry(F[cols], imp), "PC1", "median_recruitment_current")
for w, F in BRIEFF.items():
    for fs, cols in (("a", G.BRIEF_RESPONSE_A), ("a+gain", G.BRIEF_RESPONSE_A + G.BRIEF_GAIN),
                     ("b", G.BRIEF_RESPONSE_B), ("b+gain", G.BRIEF_RESPONSE_B + G.BRIEF_GAIN)):
        for imp in ("mean", "median"):
            GEO[("brief", fs, imp, w)] = G.orient(G.pca_geometry(F[cols], imp), "PC1", "median_peak_response")
    for imp in ("mean", "median"):
        GEO[("base", imp, w)] = G.pca_geometry(F[G.BRIEF_TABLE], imp)
        GEO[("gain", imp, w)] = G.orient(G.pca_geometry(F[G.BRIEF_GAIN_SET], imp), "PC1", "median_peak_gain")
print("geometries computed:", len(GEO))
MAINK = [("long", "all_long_square", "b_nobins", "mean", "dvdt"), ("ramp", "a_fig3d", "mean", "dvdt"), ("brief", "b+gain", "mean", "w50")]
for k in MAINK + [("long", "all_long_square", "a_all", "mean", "dvdt")]:
    g = GEO[k]; print(k, "cells", len(g["scores"]), "features kept", len(g["features"]), "dropped for missingness", g["dropped"])
for name, F in [("long all_long_square", LONGF[("all_long_square", "zero")]), ("ramp", RAMPF["zero"]), ("brief w10", BRIEFF["w10"])]:
    print(f"\n{name}: missing fraction per feature (nonzero only)")
    m = F.isna().mean(); print(m[m > 0].round(3).to_dict())
# Main readings (assumptions, stated in the record): long pulse = every long square step (the only sweep reading whose
# largest 9 lines are the paper's 9), without the command-binned features (Figure 2D shows none among the top loadings);
# ramp = the 8 recruitment features of Figure 3D (Methods 3, "recruitment-focused"); brief response = the 10 features of
# Figure 4C plus the two gains (Results: response set includes "input-normalized response gain"), 50 ms window;
# spikes by dV/dt >= 20 V/s (the Allen pipeline's criterion, ref. 2); mean imputation (scikit-learn default, Methods 7).
MAIN = {"long": ("long", "all_long_square", "b_nobins", "mean", "dvdt"), "ramp": ("ramp", "a_fig3d", "mean", "dvdt"),
        "brief": ("brief", "b+gain", "mean", "w50"), "gain": ("gain", "mean", "w50")}

The next cell adds readings of details the paper leaves open and that the first version fixed:
the long-pulse thresholds, command bins and spike-count scale, and the brief-pulse fast-response
window.

In [ ]:
# Further readings of details the paper leaves open: the long-pulse strong-depolarization and high-spiking
# thresholds, the command bins, spike counts per second (the steps last 0.5 s or 1 s), and the brief-pulse
# fast-response window (1 and 5 ms besides the main 2 ms). Long-pulse variants use dV/dt spikes and mean imputation.
LONGVAR = {}
for fam in ("all_long_square", "suprathreshold", "LP_named"):
    S = LONGQ[LONGQ.desc.str.match(G.LONG_FAMILIES[fam])]
    var = [(f"depol {d} mV, high {h} Hz", dict(depol=d, high=h)) for d in (10, 20, 30) for h in (10, 20, 40)]
    var += [("bins 0/100/200/300/500 pA", dict(bins=[0, 100, 200, 300, 500, np.inf])),
            ("bins 0/25/50/100/200/400/800 pA", dict(bins=[0, 25, 50, 100, 200, 400, 800, np.inf])),
            ("spike counts per second", dict(per_second=True))]
    for name, kw in var:
        F = G.long_features(S, "dvdt", **kw); F = F[lab(F.index).isin(L9)]
        table = [c for c in F.columns if not c.startswith("bin_")]
        for fs, cols in (("a_all", list(F.columns)), ("b_nobins", table)):
            if "bins" in kw and fs == "b_nobins": continue  # the bins do not enter the no-bins set
            LONGVAR[(fam, fs, name)] = G.orient(G.pca_geometry(F[cols], "mean"), "PC2", "first_spiking_command")
FASTW = []
for fw in G.FAST_ALT_MS:
    for w in ("w10", "w50"):
        F = G.brief_features(QC[QC.fam == "brief"], w, fast=fw); F = F[lab(F.index).isin(L11)]
        wl = f"{w} fast {fw:g} ms"; FASTW.append(wl)
        for fs, cols in (("a", G.BRIEF_RESPONSE_A), ("a+gain", G.BRIEF_RESPONSE_A + G.BRIEF_GAIN),
                         ("b", G.BRIEF_RESPONSE_B), ("b+gain", G.BRIEF_RESPONSE_B + G.BRIEF_GAIN)):
            for imp in ("mean", "median"):
                GEO[("brief", fs, imp, wl)] = G.orient(G.pca_geometry(F[cols], imp), "PC1", "median_peak_response")
        for imp in ("mean", "median"):
            GEO[("base", imp, wl)] = G.pca_geometry(F[G.BRIEF_TABLE], imp)
print("long-pulse threshold, bin and count variants:", len(LONGVAR), "; brief windows added:", FASTW, "(w10 and w50 use the 2 ms fast window)")


## fig1c-cells-per-protocol

> About 2270 cells entered the long-pulse geometry, about 3220 the ramp geometry and 2154 the
> brief-pulse geometry. (expected 2270 ± 227, 3220 ± 322, 2154 ± 215)

In [ ]:
QC0 = sw[G._qc(sw, baseline=False)]
rows = []
for qcname, Q in (("QC with baseline criteria", QC), ("QC without baseline criteria", QC0)):
    for fam, rx in G.LONG_FAMILIES.items():
        s = Q[(Q.fam == "long") & (Q.amp > 0) & Q.desc.str.match(rx)].session.unique()
        rows.append((qcname, "long: " + fam, len(s), int(lab(s).isin(L9).sum())))
    s = Q[Q.fam == "ramp"].session.unique(); rows.append((qcname, "ramp", len(s), int(lab(s).isin(L9).sum())))
    s = Q[Q.fam == "brief"].session.unique(); rows.append((qcname, "brief: X5SP_Search_DA_0", len(s), int(lab(s).isin(L11).sum())))
T = pd.DataFrame(rows, columns=["qc", "protocol reading", "all cells", "cells in the paper's lines"])
exp = {"long": (2270, 227), "ramp": (3220, 322), "brief": (2154, 215)}
for col in ["all cells", "cells in the paper's lines"]:
    T["within (" + col + ")"] = [abs(n - exp[p.split(":")[0]][0]) <= exp[p.split(":")[0]][1] for p, n in zip(T["protocol reading"], T[col])]
print(T.to_string())
# Where the long-pulse cells come from: stimulus set, sweep family and quality-control failures per sweep, and
# per cell, whether it has a passing long-pulse sweep and a passing ramp sweep (cells in the 9 lines).
L9cells = set(LINE[LINE.isin(L9)].index)
A = sw[(sw.fam == "long") & ((sw.amp > 0) | sw.amp.isna()) & sw.session.isin(L9cells)].copy()
A["stimulus set"] = np.where(A.desc.str.startswith("C1"), "older (C1)", "newer (X)")
A["reason"] = np.select([~A.qc_stim.fillna(False).astype(bool), ~A.qc_len.fillna(False).astype(bool),
                         ~A.base.between(G.BASE_LO, G.BASE_HI), ~(A.noise < G.BASE_SD)],
                        ["no step found", "incomplete trace", "baseline outside -100 to -40 mV", "baseline SD >= 1 mV"], "passes")
print("\nlong-pulse sweeps of cells in the 9 lines, by stimulus description and quality-control result:")
print(A.groupby(["stimulus set", "desc"]).reason.value_counts().unstack(fill_value=0).to_string())
pc = A.groupby("session").agg(sets=("stimulus set", lambda x: " + ".join(sorted(set(x)))), passing=("reason", lambda x: int((x == "passes").sum())))
pc["has a passing long sweep"] = pc.passing > 0
pc["has a passing ramp sweep"] = pc.index.isin(QC[QC.fam == "ramp"].session.unique())
print("\ncells in the 9 lines with any long-pulse sweep:", len(pc), "; with none:", len(L9cells - set(pc.index)))
print(pd.crosstab(pc.sets, [pc["has a passing long sweep"], pc["has a passing ramp sweep"]]).to_string())
rq = set(QC[QC.fam == "ramp"].session.unique()) & L9cells; lq = set(LONGQ.session.unique()) & L9cells
print("cells in the 9 lines: passing ramp", len(rq), "; passing long", len(lq), "; both", len(rq & lq), "; ramp only", len(rq - lq), "; long only", len(lq - rq))


**Result: partially reproduced.** Restricted to the driver lines the paper shows, the ramp geometry holds 3275 cells (expected 3220 ± 322) and the brief-pulse geometry 2166 (expected 2154 ± 215), both within tolerance; counted over all cells (4431 and 2595) neither is. No long-pulse reading gives a count within 2270 ± 227: every long square step gives 3276 cells in the 9 lines, the suprathreshold family 1957, and the steps named "LP" 1881. Dropping the baseline criteria changes no count by more than 2 cells. Every cell of the 9 lines has long-pulse sweeps, and the cells with a passing long-pulse sweep (3276) and those with a passing ramp sweep (3275) are the same cells but one; 1395 of them have only the older stimulus set (C1) and 1882 the newer one (X). Incomplete traces fall mostly on X3LP_Rheo (1791 sweeps) and X4PS_SupraThresh (1201), but they remove few cells, so no sweep-level criterion here gives the paper's 2270.


## fig1d-driver-line-coverage

> The long-pulse and ramp geometries each hold 9 driver lines and the brief-pulse geometry 11, with a
> subset of lines shared across protocols used for cross-protocol validation. (expected 9, 9, 11 exact;
> shared 7 ± 1)

In [ ]:
counts = {}
for fam, rx in G.LONG_FAMILIES.items():
    counts["long " + fam] = lab(QC[(QC.fam == "long") & (QC.amp > 0) & QC.desc.str.match(rx)].session.unique()).value_counts()
counts["ramp"] = lab(QC[QC.fam == "ramp"].session.unique()).value_counts()
counts["brief"] = lab(QC[QC.fam == "brief"].session.unique()).value_counts()
C = pd.DataFrame(counts).fillna(0).astype(int).sort_values("ramp", ascending=False)
print(C.head(18).to_string())
for name, named in (("long all_long_square", L9), ("long suprathreshold", L9), ("long LP_named", L9), ("ramp", L9), ("brief", L11)):
    top = list(C[name].sort_values(ascending=False).index[:len(named)])
    srt = C[name].sort_values(ascending=False)
    k = len(named)
    print(f"{name}: the {k} largest lines equal the paper's {k}: {set(top) == set(named)}; "
          f"count of the {k}th line {srt.iloc[k-1]}, of the {k+1}th {srt.iloc[k]} ({srt.index[k]}); "
          f"named lines missing from the largest {k}: {sorted(set(named) - set(top))}")
shared = sorted(set(L9) & set(L11))
print("lines shared by the long-pulse, ramp and brief-pulse sets:", len(shared), shared)
print("smallest count of any named line: long", C.loc[L9, "long all_long_square"].min(), "ramp", C.loc[L9, "ramp"].min(), "brief", C.loc[L11, "brief"].min())
# The shared count from the data's own largest lines (not from the paper's lists): the 9 largest long-pulse lines
# (every long square step), the 9 largest ramp lines and the 11 largest brief-pulse lines.
top = {n: set(C[n].sort_values(ascending=False).index[:k]) for n, k in (("long all_long_square", 9), ("ramp", 9), ("brief", 11))}
sh = top["long all_long_square"] & top["ramp"] & top["brief"]
print("shared by the largest lines of the data:", len(sh), sorted(sh), "; equal to the paper's shared lines:", sh == set(shared))
for t in (80, 91, 100, 147, 156, 200):
    print(f"minimum {t} cells: lines kept long {int((C['long all_long_square'] >= t).sum())}, ramp {int((C['ramp'] >= t).sum())}, brief {int((C['brief'] >= t).sum())}")


**Result: reproduced (resting on an assumption about the inclusion rule).** Every line the paper names is in the data. With every long square step, and for the ramp, the 9 largest lines are exactly the paper's 9 (the 9th has 156 cells, the 10th, Chrna2, 146); for the brief pulse the 11 largest are exactly the paper's 11 (91 against 79 for Nos1, the 12th). The lines shared by all three sets number 8, within 7 ± 1. No single minimum cell count selects the named lines in every protocol, so the rule assumed is "the largest lines". Under the suprathreshold and "LP" sweep readings for the long pulse, Slc32a1 is not among the 9 largest. Taken from the data alone, the 9 largest long-pulse (every long square step) and ramp lines and the 11 largest brief-pulse lines share 8 lines, the same 8 as the paper's lists. A minimum of 147 to 156 cells keeps 9 long-pulse and 9 ramp lines but only 4 brief-pulse lines; a minimum of 80 to 91 keeps 11 brief-pulse lines but 14 long-pulse and 14 ramp lines.


## fig2a-longpulse-low-dimensional

> A few principal components capture most of the variance of the long-pulse features; PC1 explains
> about 39%, PC2 about 15%, and the first five about 76%. (expected 39 ± 3.9, 15 ± 1.5, 76 ± 7.6 %)

In [ ]:
def evr_table(prefix, exp):
    rows = []
    for k, g in GEO.items():
        if k[0] != prefix: continue
        e = g["evr"] * 100
        rows.append(list(k[1:]) + [len(g["scores"]), len(g["features"]), round(e[0], 1), round(e[1], 1), round(e[:5].sum(), 1)])
    cols = {"long": ["sweeps", "feature set", "imputation", "spikes"], "ramp": ["feature set", "imputation", "spikes"],
            "brief": ["feature set", "imputation", "window"]}[prefix]
    T = pd.DataFrame(rows, columns=cols + ["cells", "features", "PC1 %", "PC2 %", "PC1-5 %"])
    T["all within"] = [all(abs(v - e) <= t for v, (e, t) in zip(r, exp)) for r in T[["PC1 %", "PC2 %", "PC1-5 %"]].to_numpy()]
    return T
T = evr_table("long", [(39, 3.9), (15, 1.5), (76, 7.6)])
print(T.to_string()); print("readings with all three within tolerance:", int(T["all within"].sum()), "of", len(T))
rows = []
for (fam, fs, name), g in LONGVAR.items():
    e = g["evr"] * 100
    rows.append([fam, fs, name, len(g["features"]), round(e[0], 1), round(e[1], 1), round(e[:5].sum(), 1),
                 abs(e[0] - 39) <= 3.9 and abs(e[1] - 15) <= 1.5 and abs(e[:5].sum() - 76) <= 7.6])
TV = pd.DataFrame(rows, columns=["sweeps", "feature set", "variant (dV/dt, mean imputation)", "features", "PC1 %", "PC2 %", "PC1-5 %", "all within"])
print("\nthreshold, bin and count variants:"); print(TV.to_string())
print("variants with all three within tolerance:", int(TV["all within"].sum()), "of", len(TV), "; PC2 range", TV["PC2 %"].min(), "to", TV["PC2 %"].max(),
      "; PC1-5 range", TV["PC1-5 %"].min(), "to", TV["PC1-5 %"].max())


**Result: not reproduced.** None of the 24 readings has all three values within tolerance. Under the main reading (every long square step, no command-binned features, mean imputation, dV/dt spikes), PC1 explains 41.9% (expected 39 ± 3.9), PC2 18.8% (15 ± 1.5) and PC1 to PC5 85.2% (76 ± 7.6). PC1 is within tolerance under several readings, while PC2 lies between 18.7 and 25.3% under every reading, above its tolerance. Varying the strong-depolarization threshold (10, 20, 30 mV), the high-spiking threshold (10, 20, 40 Hz), the command bins (two other sets) and the scale of the spike counts (per second, so that 0.5 s and 1 s steps agree) gives 66 more readings over the three sweep families; none has all three values within tolerance, PC2 lies between 18.8 and 26.1% and PC1 to PC5 between 73.4 and 87.2%.


## fig2b-longpulse-continuous-biased

> In the long-pulse PC1 to PC2 embedding cells form a continuum, not discrete clusters, and driver
> lines occupy biased regions of it.

In [ ]:
from sklearn.metrics import silhouette_score
def bimodality(x):
    from scipy.stats import skew, kurtosis
    n = len(x); g = skew(x); k = kurtosis(x)
    return (g ** 2 + 1) / (k + 3 * (n - 1) ** 2 / ((n - 2) * (n - 3)))
def bias_test(geo, lines, claim, npcs=(None, 5, 2), n_perm=1000):
    S0 = geo["scores"]; labs = lab(S0.index)
    codes = pd.Categorical(labs, categories=lines).codes
    out = []
    for npc in npcs:
        S = S0.to_numpy() if npc is None else S0.iloc[:, :npc].to_numpy()
        obs = G.mean_centroid_distance(S, codes)
        res = []
        for off in (0, 1000, 2000, 3000, 4000):
            rng = np.random.default_rng(seed(claim) + off)
            null = np.array([G.mean_centroid_distance(S, rng.permutation(codes)) for _ in range(n_perm)])
            res.append(((null >= obs).sum() / n_perm, null.mean(), null.max()))
        out.append(("all" if npc is None else npc, round(obs, 3), res[0][0], round(res[0][1], 3), round(res[0][2], 3), [r[0] for r in res]))
    return pd.DataFrame(out, columns=["PCs", "observed mean centroid distance", "perm. fraction >= observed", "null mean", "null max", "fraction, five seeds"])
def continuity(geo, lines, title):
    S = geo["scores"]; labs = lab(S.index)
    print(title, "cells", len(S))
    print("bimodality coefficient (5/9 = 0.556 marks bimodality): PC1", round(bimodality(S.PC1.to_numpy()), 3), "PC2", round(bimodality(S.PC2.to_numpy()), 3))
    sil = silhouette_score(S[["PC1", "PC2"]].to_numpy(), labs.to_numpy(), sample_size=min(2000, len(S)), random_state=0)
    print("silhouette of driver-line labels in PC1-PC2 (a measure of line overlap, not of continuity; near 0 or below means overlapping lines):", round(sil, 3))
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    for l in lines:
        m = (labs == l).to_numpy(); ax[0].scatter(S.PC1[m], S.PC2[m], s=3, alpha=0.5, label=l)
    ax[0].legend(markerscale=4, fontsize=7); ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2"); ax[0].set_title(title)
    ax[1].hist2d(S.PC1, S.PC2, bins=60, cmap="Greys", cmin=1); ax[1].set_title("cell density, PC1-PC2")
    plt.show()
    print("PC1 range", round(S.PC1.min(), 2), round(S.PC1.max(), 2), "; PC2 range", round(S.PC2.min(), 2), round(S.PC2.max(), 2), "(all points shown)")
    print("per-line median PC1, PC2:"); print(S[["PC1", "PC2"]].groupby(labs).median().round(2).T.to_string())
continuity(GEO[MAIN["long"]], L9, "long pulse, main reading")
print(bias_test(GEO[MAIN["long"]], L9, "fig2b-longpulse-continuous-biased").to_string())

**Result: reproduced (main reading).** The PC1 to PC2 embedding forms one cloud: the bimodality coefficients of PC1 and PC2 are 0.533 and 0.444, both below 0.556, the scatter and density plots show a single cloud with no gap, and the silhouette of the driver-line labels is -0.185, so lines overlap (the silhouette measures overlap of lines, not continuity). Driver lines sit in biased regions: the observed mean centroid distance (1.896 over all PCs, 1.845 over 5, 1.652 over 2) exceeds every one of 1000 label permutations (null maximum 0.491, 0.459, 0.413), under each of the five seeds.

## fig2d-longpulse-loadings

> Long-pulse PC1 is dominated by sustained firing features (firing rates, spike counts, high-spiking
> and spiking fraction) with subthreshold fraction of opposite sign; PC2 has positive loadings from
> first-spiking current, maximum firing rate, maximum spike count, subthreshold fraction and
> firing-rate slope, and negative ones from spiking fraction and baseline variability.

In [ ]:
FIG2D1 = ["mean_fr", "high_spiking_fraction", "median_fr", "median_spike_count", "spiking_fraction", "strong_depol_fraction", "max_spike_count", "subthreshold_fraction"]
FIG2D2 = ["first_spiking_command", "max_fr", "max_spike_count", "subthreshold_fraction", "fr_slope", "strong_depol_fraction", "spiking_fraction", "median_baseline_noise"]
def check_long(g):
    Ld = g["loadings"]
    t1 = list(Ld.PC1.abs().sort_values(ascending=False).index[:8]); t2 = list(Ld.PC2.abs().sort_values(ascending=False).index[:8])
    o1 = len(set(t1) & set(FIG2D1)); o2 = len(set(t2) & set(FIG2D2))
    s = np.sign(Ld.PC1["mean_fr"])
    pos1 = ["mean_fr", "median_fr", "max_fr", "median_spike_count", "max_spike_count", "high_spiking_fraction", "spiking_fraction"]
    sign1 = all(np.sign(Ld.PC1[f]) == s for f in pos1 if f in Ld.index) and np.sign(Ld.PC1["subthreshold_fraction"]) == -s
    p2 = [f for f in ["first_spiking_command", "max_fr", "max_spike_count", "subthreshold_fraction", "fr_slope"] if f in Ld.index]
    n2 = [f for f in ["spiking_fraction", "median_baseline_noise"] if f in Ld.index]
    sign2 = all(Ld.PC2[f] > 0 for f in p2) and all(Ld.PC2[f] < 0 for f in n2)
    return o1, sign1, o2, sign2, t1, t2
rows = []
for k, g in GEO.items():
    if k[0] != "long": continue
    o1, s1, o2, s2, t1, t2 = check_long(g)
    rows.append(list(k[1:]) + [o1, s1, o1 >= 6 and s1, o2, s2, o2 >= 6 and s2])
T = pd.DataFrame(rows, columns=["sweeps", "feature set", "imputation", "spikes", "PC1 overlap/8", "PC1 signs", "PC1 holds", "PC2 overlap/8", "PC2 signs", "PC2 holds"])
print(T.to_string())
g = GEO[MAIN["long"]]
print("\nmain reading, PC1 and PC2 loadings sorted by |loading| (PC2 oriented so first_spiking_command > 0):")
print(g["loadings"][["PC1", "PC2"]].reindex(g["loadings"].PC1.abs().sort_values(ascending=False).index).round(3).to_string())
print("\nmain reading, top 8 PC2:", g["loadings"].PC2.reindex(g["loadings"].PC2.abs().sort_values(ascending=False).index[:8]).round(3).to_dict())
rows = []
for (fam, fs, name), g in LONGVAR.items():
    o1, s1, o2, s2, t1, t2 = check_long(g)
    rows.append([fam, fs, name, o1, s1, o1 >= 6 and s1, o2, s2, o2 >= 6 and s2])
TV2 = pd.DataFrame(rows, columns=["sweeps", "feature set", "variant", "PC1 overlap/8", "PC1 signs", "PC1 holds", "PC2 overlap/8", "PC2 signs", "PC2 holds"])
print("\nthreshold, bin and count variants:"); print(TV2.to_string())
print("variants where both PCs hold:", int((TV2["PC1 holds"] & TV2["PC2 holds"]).sum()), "of", len(TV2))


**Result: reproduced (resting on the main reading).** Under the main reading, 6 of the 8 PC1 features of Figure 2D1 are among the 8 largest PC1 loadings, with firing-rate, spike-count, high-spiking and spiking-fraction features of one sign and subthreshold fraction of the other. 7 of the 8 PC2 features of Figure 2D2 are among the 8 largest PC2 loadings, with first-spiking current, maximum firing rate, maximum spike count, subthreshold fraction and firing-rate slope positive, and spiking fraction and baseline noise negative. Both PCs hold under all 4 readings that use every long square step without command bins, and under all 4 "LP" readings without bins; with the command-binned features included, PC2 holds in no reading, and no suprathreshold reading holds. Of the 66 threshold, bin and count variants, both PCs hold in 20: all 10 without bins on every long square step and all 10 without bins on the "LP" steps; none with the binned features and none on the suprathreshold steps.


## fig2c-longpulse-centroid-distances

> Driver-line centroids are separated in long-pulse PC space; the Pvalb centroid is farthest from Vip
> and Sncg.

In [ ]:
rows = []
for k, g in GEO.items():
    if k[0] != "long": continue
    for npc in (None, 5, 2):
        D = G.centroid_distances(G.centroids(g["scores"], lab(g["scores"].index), npc).loc[L9])
        pairs = sorted(((D.loc[a, b], a + "-" + b) for i, a in enumerate(L9) for b in L9[i + 1:]), reverse=True)
        top2 = {pairs[0][1], pairs[1][1]}
        ok = top2 == {"Pvalb-Vip", "Sncg-Pvalb"} or top2 == {"Pvalb-Vip", "Pvalb-Sncg"} or {p.replace("Sncg-Pvalb", "Pvalb-Sncg") for p in top2} == {"Pvalb-Vip", "Pvalb-Sncg"}
        rows.append(list(k[1:]) + ["all" if npc is None else npc, pairs[0][1], round(pairs[0][0], 2), pairs[1][1], round(pairs[1][0], 2), pairs[2][1], ok])
T = pd.DataFrame(rows, columns=["sweeps", "feature set", "imputation", "spikes", "PCs", "largest", "d1", "second", "d2", "third", "holds"])
print(T.to_string()); print("readings where the two largest are Pvalb-Vip and Pvalb-Sncg:", int(T.holds.sum()), "of", len(T))
D = G.centroid_distances(G.centroids(GEO[MAIN["long"]]["scores"], lab(GEO[MAIN["long"]]["scores"].index)).loc[L9])
print("\nmain reading, all PCs, centroid distances:"); print(D.round(2).to_string())

**Result: reproduced.** Under the main reading the two largest of the 36 centroid distances are Sncg to Pvalb (4.62) and Pvalb to Vip (4.02); the third is Pvalb to Sst (3.35). The criterion holds in 60 of the 72 reading and PC-count combinations. The paper's heatmap puts Pvalb to Vip first; here Sncg to Pvalb is first, which the criterion allows.

## fig2e-longpulse-pc2-pvalb-high-vip-low

> Along long-pulse PC2, Pvalb cells sit at the high end and Vip cells at the low end, with the other
> lines between.

In [ ]:
rows = []
for k, g in GEO.items():
    if k[0] != "long": continue
    for stat in ("median", "mean"):
        v = g["scores"].PC2.groupby(lab(g["scores"].index)).agg(stat).loc[L9].sort_values(ascending=False)
        rows.append(list(k[1:]) + [stat, " > ".join(v.index), v.index[0] == "Pvalb" and v.index[-1] == "Vip"])
T = pd.DataFrame(rows, columns=["sweeps", "feature set", "imputation", "spikes", "summary", "order high to low", "holds"])
print(T.to_string()); print("readings with Pvalb highest and Vip lowest:", int(T.holds.sum()), "of", len(T))
g = GEO[MAIN["long"]]
print("\nmain reading PC2 by line:"); print(g["scores"].PC2.groupby(lab(g["scores"].index)).describe().loc[L9].round(2).to_string())
rows = []
for (fam, fs, name), g in LONGVAR.items():
    v = g["scores"].PC2.groupby(lab(g["scores"].index)).median().loc[L9].sort_values(ascending=False)
    rows.append([fam, fs, name, " > ".join(v.index), v.index[0] == "Pvalb" and v.index[-1] == "Vip"])
TV3 = pd.DataFrame(rows, columns=["sweeps", "feature set", "variant", "order of medians, high to low", "holds"])
print("\nthreshold, bin and count variants (medians):"); print(TV3.to_string()); print("variants that hold:", int(TV3.holds.sum()), "of", len(TV3))


**Result: reproduced (resting on the main reading).** With PC2 oriented so that first-spiking current loads positively, Pvalb has the highest and Vip the lowest median PC2 of the 9 lines under the main reading (Pvalb median 2.26, Vip -1.04), and Pvalb highest and Vip lowest by the mean as well (2.20 and -1.06). The order holds in 15 of the 48 reading and summary combinations: all 8 without command bins on every long square step, 6 of 8 without bins on the "LP" steps, and 1 of 16 on the suprathreshold steps. With the binned features on every long square or "LP" step, Sncg is lowest. Of the 66 threshold, bin and count variants, 20 hold by the median: 10 of 10 without bins on every long square step, 9 of 10 without bins on the "LP" steps and 1 with bins.


## fig3a-ramp-variance

> The ramp feature space is low-dimensional; PC1 explains about 33%, PC2 about 21%, and the first five
> about 87% of the variance. (expected 33 ± 3.3, 21 ± 2.1, 87 ± 8.7 %)

In [ ]:
T = evr_table("ramp", [(33, 3.3), (21, 2.1), (87, 8.7)])
print(T.to_string()); print("readings with all three within tolerance:", int(T["all within"].sum()), "of", len(T))

**Result: reproduced under reading (b) (resting on the spike-detection assumption).** With all 14 ramp features and spikes detected by dV/dt, PC1 explains 36.2% (expected 33 ± 3.3), PC2 21.1% (21 ± 2.1) and PC1 to PC5 85.4% (87 ± 8.7), all within tolerance, under both imputations. With the 0 mV crossing PC1 is 38.9%, outside. Reading (a), the 8 features of Figure 3D, gives PC1 60.7 to 65.6% and PC1 to PC5 99.5 to 99.6%, far from the paper, because its recruitment-current and latency features are nearly collinear at the ramp's fixed slope of 25 pA/s. So the ramp reading under which fig3a holds is not the one under which fig3d holds in full (see "Readings that hold together").


## fig3b-ramp-continuous-biased

> In the ramp PC1 to PC2 embedding cells form a continuum without isolated driver-line clusters, and
> driver lines occupy biased regions of it.

In [ ]:
continuity(GEO[MAIN["ramp"]], L9, "ramp, main reading")
print(bias_test(GEO[MAIN["ramp"]], L9, "fig3b-ramp-continuous-biased").to_string())

**Result: reproduced (main reading).** The ramp PC1 to PC2 embedding forms one cloud (bimodality coefficients 0.488 and 0.348; the scatter and density plots show no gap; silhouette of line labels -0.22, a measure of overlap). The observed mean centroid distance (1.474 over all PCs, 1.474 over 5, 1.440 over 2) exceeds all 1000 label permutations (null maximum 0.371, 0.371, 0.355) under each of five seeds.

## fig3d-ramp-loadings

> Ramp PC1 is dominated by recruitment-current and first-spike-latency features (minimum and median
> recruitment current, first-spike latency); PC2 by firing-rate and spike-count output, with
> recruitment and latency features of opposite sign.

In [ ]:
REC = ["min_recruitment_current", "median_recruitment_current", "min_first_spike_latency", "median_first_spike_latency"]
OUT = ["median_spike_count", "max_spike_count", "median_fr", "max_fr"]
rows = []
for k, g in GEO.items():
    if k[0] != "ramp": continue
    Ld = g["loadings"]
    t1 = list(Ld.PC1.abs().sort_values(ascending=False).index[:4])
    pc1 = set(t1) == set(REC)
    t2 = list(Ld.PC2.abs().sort_values(ascending=False).index[:4])
    so = np.sign(Ld.PC2[OUT]); sr = np.sign(Ld.PC2[[f for f in REC if f in Ld.index]])
    pc2 = set(t2) == set(OUT) and (so == so.iloc[0]).all() and (sr == -so.iloc[0]).all()
    rows.append(list(k[1:]) + [", ".join(t1), pc1, ", ".join(t2), pc2])
T = pd.DataFrame(rows, columns=["feature set", "imputation", "spikes", "PC1 top 4", "PC1 holds", "PC2 top 4", "PC2 holds"])
print(T.to_string())
for k in [MAIN["ramp"], ("ramp", "a_fig3d", "mean", "zero")]:
    print("\n", k); print(GEO[k]["loadings"][["PC1", "PC2"]].round(3).to_string())

**Result: reproduced (resting on the spike-detection assumption).** On the 8 features of Figure 3D with dV/dt spike detection, the 4 recruitment-current and latency features are the 4 largest PC1 loadings (0.420 to 0.421) and the 4 firing-rate and spike-count features are the 4 largest PC2 loadings (0.281 to 0.533), with the recruitment and latency features of opposite sign (-0.248 to -0.253). With the 0 mV crossing PC1 still holds but PC2 does not: the latency loadings (-0.276) exceed that of maximum firing rate (0.244). With all 14 ramp features PC1 holds and PC2 does not.

## fig3e-ramp-recruitment-current-order

> Pvalb cells need the highest ramp recruitment current, Vip and Sncg cells the lowest, and the other
> lines lie between.

In [ ]:
for spike, F in RAMPF.items():
    v = F.median_recruitment_current
    s = v.groupby(lab(v.index)).agg(["median", "count", lambda x: x.isna().sum()]).loc[L9]
    s.columns = ["median (pA)", "cells", "cells never recruited"]
    s = s.sort_values("median (pA)", ascending=False)
    print(f"spike detection {spike}:"); print(s.round(1).to_string())
    print("holds:", s.index[0] == "Pvalb" and set(s.index[-2:]) == {"Vip", "Sncg"}, "\n")

**Result: reproduced.** Under both spike readings Pvalb has the highest line median recruitment current (280.0 pA) and Vip (61.0 pA with dV/dt, 62.4 with 0 mV) and Sncg (79.1, 80.1) the two lowest; the other six lines lie between 129.6 and 190.7 pA. Cells never recruited by the ramp (no spike) have no value: 55 of 450 Pvalb cells under dV/dt.

## fig4a-brief-response-variance

> The brief-pulse response-focused feature space is low-dimensional; PC1 explains about 47%, PC2
> about 19%, and the first five about 95% of the variance. (expected 47 ± 4.7, 19 ± 1.9, 95 ± 9.5 %)

In [ ]:
T = evr_table("brief", [(47, 4.7), (19, 1.9), (95, 9.5)])
print(T.to_string()); print("readings with all three within tolerance:", int(T["all within"].sum()), "of", len(T))

**Result: reproduced (resting on assumptions about the feature set and the response window).** On the 10 features of Figure 4C with the two gains (reading b with gain), a 50 ms response window and a 2 ms fast-response window, PC1 explains 47.8% (expected 47 ± 4.7), PC2 18.1% (19 ± 1.9) and PC1 to PC5 92.2% (95 ± 9.5). Of the 48 readings (four feature sets, two response windows, fast-response windows of 1, 2 and 5 ms, two imputations), the 6 under which all three are within tolerance are this feature set with the 50 ms window, under every fast-response window (1 ms: 47.5, 17.9, 92.1%; 5 ms: 48.5, 18.1, 92.3%) and both imputations, so the fast-response window does not change the outcome. With a 10 ms window the same set gives PC1 52.9 to 53.4%, outside; reading (a) gives PC2 25.3 to 33.8%, outside.


## fig4b-brief-continuous-biased

> In the brief-pulse response embedding cells occupy a continuous space with biased driver-line
> occupancy, and driver-line centroids are separated.

In [ ]:
continuity(GEO[MAIN["brief"]], L11, "brief pulse response, main reading")
print(bias_test(GEO[MAIN["brief"]], L11, "fig4b-brief-continuous-biased").to_string())
D = G.centroid_distances(G.centroids(GEO[MAIN["brief"]]["scores"], lab(GEO[MAIN["brief"]]["scores"].index)).loc[L11])
print("centroid distances, main reading, all PCs: min", round(D.to_numpy()[np.triu_indices(11, 1)].min(), 3), "max", round(D.to_numpy().max(), 3))

**Result: reproduced (main reading).** The brief-pulse response embedding forms one cloud (bimodality coefficients 0.325 and 0.195; the scatter and density plots show no gap; silhouette of line labels -0.14, a measure of overlap). The observed mean centroid distance over the 11 lines (1.747 over all PCs, 1.685 over 5, 1.279 over 2) exceeds all 1000 label permutations (null maximum 0.540, 0.522, 0.453) under each of five seeds, and the 55 centroid distances range from 0.384 to 4.138.

## fig4c-brief-response-pc1-loadings

> Brief response PC1 is dominated by fast and total voltage-response amplitude features (fast response
> mean, response mean, response peak, response quantiles, fast peak response), all loading with the
> same sign.

In [ ]:
AMP = set(G.BRIEF_RESPONSE_A + G.BRIEF_RESPONSE_B)
rows = []
for k, g in GEO.items():
    if k[0] != "brief": continue
    Ld = g["loadings"].PC1
    order = list(Ld.abs().sort_values(ascending=False).index)
    top5 = order[:5]
    amp_feats = [f for f in order if f in AMP]
    rows.append(list(k[1:]) + [", ".join(top5), all(f in AMP for f in order[:3]), all(f in AMP for f in top5),
                 len(set(np.sign(Ld[top5]))) == 1, len(set(np.sign(Ld[amp_feats]))) == 1,
                 all(f in AMP for f in top5) and len(set(np.sign(Ld[top5]))) == 1])
T = pd.DataFrame(rows, columns=["feature set", "imputation", "window", "PC1 top 5", "top 3 amplitude", "top 5 amplitude", "top 5 same sign",
                                "every amplitude feature same sign", "holds (top 5 amplitude, one sign)"])
print(T.to_string())
print("\nmain reading PC1 loadings:"); print(GEO[MAIN["brief"]]["loadings"].PC1.sort_values().round(3).to_string())

**Result: reproduced (resting on the feature-set assumption).** Under the main reading the five largest PC1 loadings are mean response (median and mean over sweeps), median fast-response mean, median peak response and median fast peak, all of one sign, and every amplitude feature loads with that sign; the gains load less (0.188 and 0.241). The same holds for all 24 readings (b), with or without gain and under every fast-response window. Under reading (a), maximum fast peak and maximum peak response load with the opposite sign, and with gain added, median peak gain is among the top five.

## fig4e-brief-response-base-alignment

> Across 2154 cells, brief response PC1 scores correlate strongly with PC2 of the broader brief-pulse
> base geometry (Pearson r = 0.875, Spearman r = 0.900). (expected 0.875 ± 0.0875, 0.900 ± 0.09, n
> 2154 ± 215)

In [ ]:
rows = []
for k, g in GEO.items():
    if k[0] != "brief": continue
    base = GEO[("base", k[2], k[3])]
    r1 = g["scores"].PC1; B = base["scores"].loc[r1.index]
    pr = [abs(pearsonr(r1, B[f"PC{i}"])[0]) for i in range(1, 6)]
    sr = abs(spearmanr(r1, B.PC2)[0])
    best = int(np.argmax(pr)) + 1
    rows.append(list(k[1:]) + [len(r1), round(pr[1], 3), round(sr, 3), f"PC{best}", round(pr[best - 1], 3), round(abs(spearmanr(r1, B[f'PC{best}'])[0]), 3),
                 abs(pr[1] - 0.875) <= 0.0875 and abs(sr - 0.9) <= 0.09])
T = pd.DataFrame(rows, columns=["feature set", "imputation", "window", "cells", "|Pearson| PC2", "|Spearman| PC2", "best base PC", "|Pearson| best", "|Spearman| best", "PC2 within"])
print(T.to_string())
b = GEO[("base", "mean", "w10")]
print("\nbase geometry (mean, w10): features", len(b["features"]), "dropped", b["dropped"], "EVR % PC1-5", (b["evr"][:5] * 100).round(1))
print(b["loadings"][["PC1", "PC2", "PC3"]].round(3).to_string())

**Result: reproduced (resting on the main reading).** Across 2166 cells (expected 2154 ± 215), response PC1 under the main reading correlates with base PC2 with |Pearson r| = 0.886 (expected 0.875 ± 0.0875) and |Spearman r| = 0.876 (0.900 ± 0.09), and base PC2 is the base axis that matches best. All 24 readings (b) are within tolerance (|Pearson r| 0.879 to 0.911); readings (a) give |Pearson r| of 0.95 to 0.99, within tolerance in 8 of 24 (reading a without gain, at 10 ms under every fast window and at 50 ms with the 1 ms fast window).

## fig4f-brief-gain-vip-cck-high

> On peak response gain the driver-line order differs from long-pulse and ramp; Vip and Cck cells have
> the highest gain, and Pvalb cells are not at the upper extreme.

In [ ]:
GAIN = {}
for w, F in BRIEFF.items():
    v = F.median_peak_gain
    s = v.groupby(lab(v.index)).agg(["median", "count"]).loc[L11].sort_values("median", ascending=False)
    GAIN[w] = s
    print(f"window {w}: line medians of the per-cell median peak-response gain (mV/pA)")
    print(s.round(4).to_string())
    print("Vip and Cck the two highest:", set(s.index[:2]) == {"Vip", "Cck"}, "; Pvalb below the median of line medians:",
          s.loc["Pvalb", "median"] < s["median"].median(), f"(Pvalb {s.loc['Pvalb', 'median']:.4f}, median of medians {s['median'].median():.4f})\n")
F = BRIEFF["w10"]
ns = QC[QC.fam == "brief"].groupby("session").apply(lambda g: (g.n_zero == 0).sum())
print("variant: gain over sweeps without a spike only (w10)")
sub = QC[(QC.fam == "brief") & (QC.n_zero == 0)]
gv = (sub.peak_w10 / sub.amp).groupby(sub.session).median()
gv = gv[lab(gv.index).isin(L11)]
print(gv.groupby(lab(gv.index)).median().sort_values(ascending=False).round(4).to_string())

**Result: reproduced.** Under both windows (which give identical line medians), Cck (0.0855 mV/pA) and Vip (0.0805) have the two highest line medians of the 11 lines, and Pvalb's median (0.0477) is below the median of line medians (0.0566), tenth of eleven. Restricting gain to sweeps without a spike gives the same two at the top and Pvalb tenth.

## disc-brief-gain-sncg-cck-non-pvalb

> Cck and Sncg cells also lie toward the high-gain end of the brief-pulse distribution, so high
> transient sensitivity is spread over several non-Pvalb inhibitory groups.

In [ ]:
for w, s in GAIN.items():
    m = s["median"]; upper = set(m.sort_values(ascending=False).index[:5])  # upper half of 11 (5 above the middle line)
    rank = {l: i + 1 for i, l in enumerate(m.sort_values(ascending=False).index)}
    ok = all(m[l] > m["Pvalb"] for l in ("Cck", "Sncg")) and all(rank[l] <= 5 for l in ("Cck", "Sncg"))
    print(f"window {w}: rank (1 = highest) Cck {rank['Cck']}, Sncg {rank['Sncg']}, Pvalb {rank['Pvalb']}; "
          f"Cck {m['Cck']:.4f}, Sncg {m['Sncg']:.4f}, Pvalb {m['Pvalb']:.4f}; holds (both above Pvalb and in the top 5 of 11): {ok}")

**Result: reproduced.** Cck ranks first (0.0855 mV/pA) and Sncg third (0.0618) of the 11 line medians, both above Pvalb (0.0477, tenth), under both windows.

## fig5a-longpc2-ramppc1-centroids

> Across the 9 driver lines, centroids on long-pulse PC2 and ramp recruitment PC1 are closely aligned
> (Spearman r = 0.883, p = 0.002), with Pvalb high and Vip low on both. (expected r 0.883 ± 0.088)

In [ ]:
rows = []
for kl, gl in GEO.items():
    if kl[0] != "long": continue
    cl = gl["scores"].PC2.groupby(lab(gl["scores"].index)).mean().loc[L9]
    for kr, gr in GEO.items():
        if kr[0] != "ramp": continue
        cr = gr["scores"].PC1.groupby(lab(gr["scores"].index)).mean().loc[L9]
        r, p = spearmanr(cl, cr)
        rows.append([" ".join(kl[1:]), " ".join(kr[1:]), round(r, 3), p, cl.idxmax(), cl.idxmin(), cr.idxmax(), cr.idxmin(), abs(r - 0.883) <= 0.088 and p < 0.05])
T = pd.DataFrame(rows, columns=["long reading", "ramp reading", "Spearman r", "p", "long max", "long min", "ramp max", "ramp min", "holds"])
print(T.to_string()); print("combinations that hold:", int(T.holds.sum()), "of", len(T), "; r range", T["Spearman r"].min(), "to", T["Spearman r"].max())
gl, gr = GEO[MAIN["long"]], GEO[MAIN["ramp"]]
cl = gl["scores"].PC2.groupby(lab(gl["scores"].index)).mean().loc[L9]; cr = gr["scores"].PC1.groupby(lab(gr["scores"].index)).mean().loc[L9]
print("\nmain readings, centroids:"); print(pd.DataFrame({"long PC2": cl, "ramp PC1": cr}).round(3).to_string())
r, p = spearmanr(cl, cr); print("Spearman r", round(r, 3), "p", round(p, 4), "; Pearson r", round(pearsonr(cl, cr)[0], 3))

**Result: reproduced (resting on the main readings).** Under the main readings, the 9 line centroids on long-pulse PC2 and ramp PC1 give Spearman r = 0.900 (expected 0.883 ± 0.088), p = 0.0009. Pvalb is highest on both axes; Vip is lowest on long-pulse PC2 and second lowest on ramp PC1, where Sncg is lowest (-1.498 against Vip -1.393), so the statement's "Vip low on both" holds only loosely on the ramp. Across the 192 combinations of long-pulse and ramp readings (the "LP" sweeps now included), r ranges from -0.317 to 1.0 and 80 hold.

## fig5b-long-ramp-distance-alignment

> Pairwise driver-line centroid distances in the long-pulse and ramp recruitment geometries are
> positively correlated over 36 line pairs (Spearman r = 0.707, p < 0.001). (expected r 0.707 ± 0.071,
> 36 pairs)

In [ ]:
# Mantel test (variant): the 36 distances come from one matrix and are not independent, so the line labels of
# the ramp matrix are permuted (10000 permutations, the same for every combination) and Spearman r recomputed.
from scipy.stats import rankdata
rngM = np.random.default_rng(seed("fig5b-long-ramp-distance-alignment"))
PERM = np.array([rngM.permutation(9) for _ in range(10000)])
iu = np.triu_indices(9, 1)
def mantel(dl, Dr):
    rl = rankdata(dl); rl = (rl - rl.mean()) / rl.std()
    Rr = Dr[PERM[:, iu[0]], PERM[:, iu[1]]]
    Rr = rankdata(Rr, axis=1); Rr = (Rr - Rr.mean(1, keepdims=True)) / Rr.std(1, keepdims=True)
    null = (Rr * rl).mean(1); obs = spearmanr(dl, Dr[iu])[0]
    return ((null >= obs).sum() + 1) / (len(null) + 1)
rows = []
for kl, gl in GEO.items():
    if kl[0] != "long": continue
    for kr, gr in GEO.items():
        if kr[0] != "ramp": continue
        for npc in (None, 5, 2):
            dl = G.upper(G.centroid_distances(G.centroids(gl["scores"], lab(gl["scores"].index), npc)), L9)
            dr = G.upper(G.centroid_distances(G.centroids(gr["scores"], lab(gr["scores"].index), npc)), L9)
            r, p = spearmanr(dl, dr)
            Dr = G.centroid_distances(G.centroids(gr["scores"], lab(gr["scores"].index), npc)).loc[L9, L9].to_numpy()
            rows.append([" ".join(kl[1:]), " ".join(kr[1:]), "all" if npc is None else npc, len(dl), round(r, 3), p, mantel(dl, Dr), round(pearsonr(dl, dr)[0], 3), abs(r - 0.707) <= 0.071 and p < 0.001])
T5b = pd.DataFrame(rows, columns=["long reading", "ramp reading", "PCs", "pairs", "Spearman r", "p", "Mantel p (variant)", "Pearson r (variant)", "holds"])
print(T5b.to_string()); print("combinations that hold:", int(T5b.holds.sum()), "of", len(T5b), "; Spearman r range", T5b["Spearman r"].min(), "to", T5b["Spearman r"].max())
print(T5b.groupby("PCs")[["Spearman r"]].describe().round(3).to_string())
m = T5b[(T5b["long reading"] == " ".join(MAIN["long"][1:])) & (T5b["ramp reading"] == " ".join(MAIN["ramp"][1:]))]
print("\nmain readings:"); print(m.to_string())
print("\nMantel p over all combinations: max", T5b["Mantel p (variant)"].max(), "; combinations with Mantel p < 0.001:", int((T5b["Mantel p (variant)"] < 0.001).sum()), "of", len(T5b))
print("\ncombinations that hold:"); print(T5b[T5b.holds].to_string())


**Result: reproduced under the suprathreshold long-pulse reading (resting on it).** Under the main readings the 36 pairwise centroid distances correlate positively and strongly across the two geometries (Spearman r 0.881 over all PCs, 0.878 over 5, 0.877 over 2; p near 1e-12; Pearson 0.921), above the paper's 0.707 ± 0.071. Across the 576 combinations of readings (the "LP" sweeps now included) r ranges from 0.447 to 0.969, and 32 have r within tolerance with p < 0.001, all with the suprathreshold long-pulse sweeps (r 0.637 to 0.667, Spearman p about 1e-5); 16 of them use dV/dt spikes in both geometries. The criterion holds under a reading, so the claim is reproduced, but only on the sweeps under which fig2d and fig2e fail and fig5a does not hold (see "Readings that hold together"). The 36 distances are not independent; a Mantel test that permutes the ramp line labels gives p = 0.0001 under the main readings but 0.0088 to 0.0213 for the 32 combinations within tolerance, so with that test none of them reaches p < 0.001.


## Readings that hold together

Each claim above is judged under the readings its criterion names. This section shows which claims
hold together when one reading is taken for each geometry (long pulse and ramp, with one spike
detection and one imputation for both).

In [ ]:
# Which claims hold together under one reading of each geometry. Spike detection and imputation are the same
# in the long-pulse and ramp geometries (one choice of each for the whole analysis); each claim is judged by its
# own criterion, over the PC counts and per-line summaries that criterion names.
def f2a(g): e = g["evr"] * 100; return abs(e[0] - 39) <= 3.9 and abs(e[1] - 15) <= 1.5 and abs(e[:5].sum() - 76) <= 7.6
def f2d(g): o1, s1, o2, s2, _, _ = check_long(g); return o1 >= 6 and s1 and o2 >= 6 and s2
def f2c(g):
    ok = False
    for npc in (None, 5, 2):
        D = G.centroid_distances(G.centroids(g["scores"], lab(g["scores"].index), npc).loc[L9])
        pr = sorted(((D.loc[a, b], frozenset((a, b))) for i, a in enumerate(L9) for b in L9[i + 1:]), key=lambda x: -x[0])
        ok |= {pr[0][1], pr[1][1]} == {frozenset(("Pvalb", "Vip")), frozenset(("Pvalb", "Sncg"))}
    return ok
def f2e(g):
    return any((lambda v: v.index[0] == "Pvalb" and v.index[-1] == "Vip")(g["scores"].PC2.groupby(lab(g["scores"].index)).agg(st).loc[L9].sort_values(ascending=False)) for st in ("median", "mean"))
def f3a(g): e = g["evr"] * 100; return abs(e[0] - 33) <= 3.3 and abs(e[1] - 21) <= 2.1 and abs(e[:5].sum() - 87) <= 8.7
def f3d(g):
    Ld = g["loadings"]; pc1 = set(Ld.PC1.abs().sort_values(ascending=False).index[:4]) == set(REC)
    so = np.sign(Ld.PC2[OUT]); sr = np.sign(Ld.PC2[REC])
    pc2 = set(Ld.PC2.abs().sort_values(ascending=False).index[:4]) == set(OUT) and (so == so.iloc[0]).all() and (sr == -so.iloc[0]).all()
    return "reproduced" if pc1 and pc2 else "partial" if pc1 or pc2 else "no"
def f5a(gl, gr):
    r, p = spearmanr(gl["scores"].PC2.groupby(lab(gl["scores"].index)).mean().loc[L9], gr["scores"].PC1.groupby(lab(gr["scores"].index)).mean().loc[L9])
    return abs(r - 0.883) <= 0.088 and p < 0.05
def f5b(gl, gr):
    for npc in (None, 5, 2):
        r, p = spearmanr(G.upper(G.centroid_distances(G.centroids(gl["scores"], lab(gl["scores"].index), npc)), L9),
                         G.upper(G.centroid_distances(G.centroids(gr["scores"], lab(gr["scores"].index), npc)), L9))
        if abs(r - 0.707) <= 0.071 and p < 0.001: return True
    return False
rows = []
for kl, gl in GEO.items():
    if kl[0] != "long": continue
    for kr, gr in GEO.items():
        if kr[0] != "ramp" or kr[2:] != kl[3:]: continue
        r = dict(long=" ".join(kl[1:3]), ramp=kr[1], imputation=kl[3], spikes=kl[4], fig2a=f2a(gl), fig2d=f2d(gl), fig2c=f2c(gl), fig2e=f2e(gl),
                 fig3a=f3a(gr), fig3d=f3d(gr), fig5a=f5a(gl, gr), fig5b=f5b(gl, gr))
        r["held (of 8)"] = sum(bool(v) and v != "no" and v != "partial" for k, v in r.items() if k.startswith("fig"))
        rows.append(r)
TJ = pd.DataFrame(rows).sort_values("held (of 8)", ascending=False)
print(TJ.to_string(index=False))
print("most claims held under one chain of readings:", TJ["held (of 8)"].max(), "of 8; chains holding fig3a and fig3d (reproduced) together:",
      int((TJ.fig3a & (TJ.fig3d == "reproduced")).sum()), "; holding fig5b with fig2d and fig2e:", int((TJ.fig5b & TJ.fig2d & TJ.fig2e).sum()),
      "; holding fig5b with fig5a:", int((TJ.fig5b & TJ.fig5a).sum()))


**Result: no single chain of readings holds every claim.** Taking one reading for the long-pulse and one for the ramp geometry, with one spike detection and one imputation for both, at most 5 of the 8 claims on these geometries hold together (fig2d, fig2c, fig2e, fig5a and either fig3a or fig3d in full), under every long square step or the "LP" steps without bins and dV/dt spikes. No chain holds fig3a together with fig3d in full, none holds fig5b together with fig2d and fig2e, none holds fig5b with fig5a, and fig2a holds in none. Each claim's outcome is judged by its own criterion, under the readings it names; this table shows that the readings that reproduce them differ between claims.


## fig5c-balanced-correlations-positive

> Under balanced resampling, distance-matrix correlations are positive for every protocol pair, with
> medians near 0.79 (long pulse against ramp recruitment) and 0.6 to 0.65 for the pairs involving
> brief response or brief gain.

In [ ]:
PAIRS = [("long", "ramp", 0.79), ("ramp", "brief", 0.65), ("ramp", "gain", 0.65), ("long", "gain", 0.62), ("brief", "gain", 0.62), ("long", "brief", 0.61)]
# Readings of the geometries entering the balanced comparisons. Long pulse: the main reading, or the suprathreshold or
# LP-named sweeps (no bins, mean, dV/dt). Ramp: the Figure 3D set (main) or all 14 features (mean, dV/dt).
LONG_ALT = {"long main": MAIN["long"], "long suprathreshold": ("long", "suprathreshold", "b_nobins", "mean", "dvdt"),
            "long LP_named": ("long", "LP_named", "b_nobins", "mean", "dvdt")}
RAMP_ALT = {"ramp main": MAIN["ramp"], "ramp all 14": ("ramp", "b_all", "mean", "dvdt")}
# Shared lines. "pairwise" (main): the lines shared by the two geometries of the pair (Methods 5, "for each pair of
# geometries, shared driver lines were identified"): 9 for long against ramp, 11 for brief response against gain,
# 8 otherwise. "all three": the 8 lines shared by all three protocols, for every pair.
def setup(ka, kb, npc, m, shared):
    ga, gb = GEO[ka], GEO[kb]
    la, lb = lab(ga["scores"].index), lab(gb["scores"].index)
    pool = L11 if {ka[0], kb[0]} <= {"brief", "gain"} else L9
    lines = [l for l in pool if l in set(la) and l in set(lb)]
    if shared == "all three": lines = [l for l in lines if l in set(L9) & set(L11)]
    ca, cb = la.value_counts(), lb.value_counts()
    mm = min(min(ca.get(l, 0), cb.get(l, 0)) for l in lines) if m == "smallest" else m
    lines = [l for l in lines if ca.get(l, 0) >= mm and cb.get(l, 0) >= mm]
    SA = ga["scores"].to_numpy() if npc is None else ga["scores"].iloc[:, :npc].to_numpy()
    SB = gb["scores"].to_numpy() if npc is None else gb["scores"].iloc[:, :npc].to_numpy()
    return SA, la.to_numpy(), SB, lb.to_numpy(), lines, mm
def settings():
    """Every combination of readings: (long, ramp, shared lines, PCs, cells per line) -> geometry key per protocol."""
    for npc in (None, 5, 2):
        for m in ("smallest", 20):
            for sh in ("pairwise", "all three"):
                for ln, kl in LONG_ALT.items():
                    for rn, kr in RAMP_ALT.items():
                        if sh == "all three" and (ln, rn) != ("long main", "ramp main"): continue
                        yield (ln, rn, sh, "all" if npc is None else npc, m), npc, m, sh, {"long": kl, "ramp": kr, "brief": MAIN["brief"], "gain": MAIN["gain"]}
ITERS = 1000
BAL = {}   # (geometry key a, geometry key b, npc, cells per line, lines) -> 1000 balanced r under the claim's seed
rows, srows = [], []
for name, npc, m, sh, keys in settings():
    meds_set = []
    for a, b, e in PAIRS:
        SA, la, SB, lb, lines, mm = setup(keys[a], keys[b], npc, m, sh)
        ck = (keys[a], keys[b], npc, mm, tuple(lines))
        if ck not in BAL:
            meds = []
            for off in (0, 1000, 2000, 3000, 4000):
                rng = np.random.default_rng(seed("fig5c-balanced-correlations-positive") + off)
                v = G.balanced(SA, la, SB, lb, lines, mm, ITERS, rng)
                if off == 0: BAL[ck] = v
                meds.append(np.median(v))
            rows.append([*name, f"{a} vs {b}", mm, len(lines), e, round(meds[0], 3), round(np.mean(BAL[ck] > 0), 3), abs(meds[0] - e) <= 0.1 * e, round(min(meds), 3), round(max(meds), 3)])
        meds_set.append(np.median(BAL[ck]))
    srows.append([*name] + [round(x, 3) for x in meds_set] + [all(x > 0 for x in meds_set), sum(abs(x - e) <= 0.1 * e for x, (_, _, e) in zip(meds_set, PAIRS))])
T5c = pd.DataFrame(rows, columns=["long", "ramp", "shared lines", "PCs", "cells per line", "pair", "m", "lines", "expected", "median r", "fraction r > 0", "within", "median, five seeds min", "max"])
print("each comparison, at the first setting that uses it:"); print(T5c.to_string())
S5c = pd.DataFrame(srows, columns=["long", "ramp", "shared lines", "PCs", "cells per line"] + [f"{a}-{b} ({e})" for a, b, e in PAIRS] + ["all positive", "n within (of 6)"])
print("\nall settings, median balanced r per pair:"); print(S5c.to_string())
print("\nsettings with every median positive:", int(S5c["all positive"].sum()), "of", len(S5c), "; with all 6 within tolerance:", int((S5c["n within (of 6)"] == 6).sum()),
      "; most within in one setting:", S5c["n within (of 6)"].max())
print("main setting (long main, ramp main, pairwise lines, all PCs, smallest count):")
print(S5c[(S5c.long == "long main") & (S5c.ramp == "ramp main") & (S5c["shared lines"] == "pairwise") & (S5c.PCs == "all") & (S5c["cells per line"] == "smallest")].T.to_string())


**Result: partially reproduced.** Under balanced resampling every median correlation is positive in all 42 settings (long-pulse sweeps of every long square step, the suprathreshold or the "LP" family; the Figure 3D or all 14 ramp features; lines shared pairwise or by all three protocols; all PCs, 5 or 2; cells per line the smallest retained count or 20). Under the main setting (main readings, pairwise shared lines, all PCs, smallest count: 156 cells over 9 lines for long against ramp, 91 over 11 lines for brief response against gain, 91 over 8 lines otherwise), the medians are 0.870 long against ramp (expected 0.79), 0.784 ramp against brief response (0.65), 0.592 ramp against gain (0.65), 0.454 long against gain (0.62), 0.795 brief response against gain (0.62) and 0.632 long against brief response (0.61); 2 of 6 are within tolerance. With the 8 lines shared by all three protocols, brief response against gain is 0.700, also outside. No setting puts more than 3 of the 6 within tolerance. The five seeds move each median by at most 0.008 under the main setting.


## fig5d-f-true-above-shuffle-null

> For every protocol pair, the balanced true distance-matrix correlations lie above the driver-label
> shuffle null distribution, which centres near zero.

In [ ]:
rows, srows = [], []
NULL = {}
for name, npc, m, sh, keys in settings():
    ok = {"one": True, "both": True}
    for a, b, e in PAIRS:
        SA, la, SB, lb, lines, mm = setup(keys[a], keys[b], npc, m, sh)
        ck = (keys[a], keys[b], npc, mm, tuple(lines))
        med = np.median(BAL[ck])
        for shf in ("one", "both"):
            if ck + (shf,) not in NULL:
                ps = []
                for off in (0, 1000, 2000, 3000, 4000):
                    rng = np.random.default_rng(seed("fig5d-f-true-above-shuffle-null") + off)
                    null = G.balanced(SA, la, SB, lb, lines, mm, ITERS, rng, shuffle=shf)
                    if off == 0: NULL[ck + (shf,)] = null
                    ps.append((null >= med).mean())
                null0 = NULL[ck + (shf,)]
                p1 = (null0 >= med).mean(); p2 = ((null0 >= med).sum() + 1) / (len(null0) + 1)
                h = (p1 < 0.05 or p2 < 0.05) and abs(np.median(null0)) <= 0.15
                rows.append([*name, f"{a} vs {b}", shf, round(med, 3), round(np.median(null0), 3), p1, round(p2, 4), h, max(ps)])
                NULL[ck + (shf, "row")] = h
            ok[shf] &= NULL[ck + (shf, "row")]
    for shf in ("one", "both"): srows.append([*name, shf, ok[shf]])
T5d = pd.DataFrame(rows, columns=["long", "ramp", "shared lines", "PCs", "cells per line", "pair", "shuffle", "true median", "null median", "p (fraction)", "p (+1)", "holds", "p (fraction), max of five seeds"])
print("each comparison, at the first setting that uses it:"); print(T5d.to_string())
S5d = pd.DataFrame(srows, columns=["long", "ramp", "shared lines", "PCs", "cells per line", "shuffle", "all six hold"])
print("\nsettings:"); print(S5d.to_string())
print("settings where all six pairs hold:", int(S5d["all six hold"].sum()), "of", len(S5d))
print("null medians range", T5d["null median"].min(), "to", T5d["null median"].max())
mk = {"long": MAIN["long"], "ramp": MAIN["ramp"], "brief": MAIN["brief"], "gain": MAIN["gain"]}
fig, axs = plt.subplots(2, 3, figsize=(13, 7))
for ax, (a, b, e) in zip(axs.ravel(), PAIRS):
    _, _, _, _, lines, mm = setup(mk[a], mk[b], None, "smallest", "pairwise"); ck = (mk[a], mk[b], None, mm, tuple(lines))
    ax.hist(BAL[ck], bins=40, alpha=0.6, color="tab:orange", label="balanced true")
    ax.hist(NULL[ck + ("both",)], bins=40, alpha=0.6, color="tab:blue", label="label shuffle (both)")
    ax.set_title(f"{a} vs {b} (main setting)"); ax.set_xlabel("Spearman r"); ax.legend(fontsize=7); ax.set_xlim(-1, 1)
plt.show()
print("all values inside the axes (-1 to 1).")


**Result: reproduced under the reading that shuffles labels in both geometries (resting on assumptions about the null).** Under the main setting the shuffle nulls centre near zero (medians -0.042 to 0.001), and with labels permuted in both geometries every pair's empirical p is below 0.05 (largest 0.038, long pulse against gain; at most 0.038 over five seeds). With labels permuted in one geometry only, long pulse against gain gives p = 0.051 (0.052 with one added), so that reading holds for 5 of 6 pairs. All six pairs hold in 24 of the 84 settings and shuffles: every setting with the "LP" long-pulse sweeps and the smallest count, the main long-pulse sweeps with the smallest count when both geometries are shuffled or with 5 PCs, and none with the suprathreshold sweeps or with 20 cells per line. Over all settings the null medians lie between -0.049 and 0.010.


## Summary

| Claim | Outcome | Rests on an assumption |
|---|---|---|
| fig1c-cells-per-protocol | partially reproduced (ramp and brief counts match; no long-pulse reading does) | yes |
| fig1d-driver-line-coverage | reproduced | yes |
| fig2a-longpulse-low-dimensional | not reproduced (no reading matches) | yes (quality control and feature definitions are this attempt's) |
| fig2b-longpulse-continuous-biased | reproduced | yes |
| fig2d-longpulse-loadings | reproduced | yes |
| fig2c-longpulse-centroid-distances | reproduced | yes |
| fig2e-longpulse-pc2-pvalb-high-vip-low | reproduced | yes |
| fig3a-ramp-variance | reproduced (reading b) | yes |
| fig3b-ramp-continuous-biased | reproduced | yes |
| fig3d-ramp-loadings | reproduced | yes |
| fig3e-ramp-recruitment-current-order | reproduced | no (both spike readings agree) |
| fig4a-brief-response-variance | reproduced | yes |
| fig4b-brief-continuous-biased | reproduced | yes |
| fig4c-brief-response-pc1-loadings | reproduced | yes |
| fig4e-brief-response-base-alignment | reproduced | yes |
| fig4f-brief-gain-vip-cck-high | reproduced | no (both windows agree) |
| disc-brief-gain-sncg-cck-non-pvalb | reproduced | no |
| fig5a-longpc2-ramppc1-centroids | reproduced | yes |
| fig5b-long-ramp-distance-alignment | reproduced (only with the suprathreshold long-pulse sweeps; r 0.88 under the main readings) | yes |
| fig5c-balanced-correlations-positive | partially reproduced (all positive, at most 3 of 6 within tolerance) | yes |
| fig5d-f-true-above-shuffle-null | reproduced (labels shuffled in both geometries) | yes |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper gives no quality-control threshold,
spike criterion, response window, feature-set membership, line inclusion rule or resampling
setting, and links no code. Each was run under the readings the claims name, and a main reading
was adopted: every long square step for the long pulse, without the command-binned features
(Figure 2D shows none among the top loadings); the 8 recruitment features of Figure 3D for the
ramp; the 10 features of Figure 4C with the two gains for the brief response, with a 50 ms window
and a 2 ms fast-response window; spikes by dV/dt of 20 V/s; mean imputation; the paper's named
lines; and, for the balanced comparisons, the lines shared by each pair of geometries. The
long-pulse thresholds, command bins and fast-response window were also varied, and change no
outcome. The readings that reproduce the claims are not the same for every claim: no single
chain of long-pulse and ramp readings holds more than 5 of the 8 claims on those geometries
(section "Readings that hold together"). The per-sweep features are
this attempt's definitions of the names in Supplementary Table 1.

What was found about the dataset. The files agree with the description: 4435 cells from 1040
mice, voltage in volts sampled at 50 kHz (127 sweeps at 200 kHz), command and voltage on one clock
(the voltage moves with the sign of the command in the 5 to 25 ms after onset and not before it, and the
ramp's recruitment current divided by latency equals the ramp slope, 25 pA/s). The driver line is
given only inside the genotype string; Sncg cells carry a Flp allele and no Cre. One recording is
deposited twice: sessions 680563483 (subject 673135474, Vip-IRES-Cre) and 680897463 (subject
673135516, Ndnf-IRES2-dgCre with Slc32a1-FlpO) have identical sweeps under two genotypes; both are
left out of the line analyses. The 1837 cells recorded only with the older stimulus set (names
beginning C1) have no X5SP_Search_DA_0 sweeps, so the brief-pulse geometry can hold at most the 2597
cells with the newer set.
Many sweeps end early, with zeros to the end of the series (13% of long-pulse sweeps fail the
complete-trace check).

In [ ]:
print(f"total run time {time.time() - t0:.0f} s; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3:.0f} MB")